Ratatouille_LLMFinalProject.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1drrUBq1dKl4ZkvypjRL4M6eTV9eDeeFk

# **RATATOUILLE LLM**
This is an AI-assistant (RAG) that acts like a chef. The inspiration comes from a daily basis struggle for people with diet restrictions: lacking ideas when cooking for themselves in order to avoid boring meals by routine. When user does not know what to cook, they can input their mood for today (what they'd like to eat today), allergies, and dislikes. This assistant will look for recipes, filter and rank them based on the criteria added, and will display a well-structured and easy-to-follow recipe.

┌────────────────┐      ┌────────────────┐      ┌────────────────┐      ┌────────────────┐
│    AGENT 1     │      │    AGENT 2     │      │    AGENT 3     │      │    AGENT 4     │
│   RESEARCHER   │─────>│    FILTER      │─────>│    RANKER      │─────>│    WRITER      │
├────────────────┤      ├────────────────┤      ├────────────────┤      ├────────────────┤
│ - Web search   │      │ - Remove       │      │ - Score by     │      │ - Format all   │
│ - Find recipes │      │   dislikes     │      │   preferences  │      │   TOP 3        │
│ - Get details  │      │ - Check        │      │ - Rank best    │      │ - Add tips     │
│                │      │   allergies    │      │   to worst     │      │ - Markdown     │
└────────────────┘      └────────────────┘      └────────────────┘      └────────────────┘
        |                      |                      |                      |
        v                      v                      v                      v
   5-10 recipes           3-5 filtered            Top 3               3 Recipe Cards


In [ ]:
%pip install crewai crewai-tools

import os
from google.colab import userdata
os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

#first agent - researcher
from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool
from IPython.display import Markdown, display

# Define the research topic
topic = "tuna"
dislikes = "bread"
allergies = "lettuce"
preferences = "no oil"

# Create the Researcher agent
researcher = Agent(
    name="Researcher",
    role="Recipe Researcher",
    goal="Find the best recipes based on user's ingredients preferences",
    backstory="Expert culinary researcher who finds the best recipes online",
    tools=[SerperDevTool()],
    verbose=True
)
# Define tasks
research_task = Task(
    description=f"""Find 10 different recipes featuring {topic} as the main ingredient.
    For EACH recipe, you MUST provide ALL of the following:

    1. *Recipe Name*: Creative, descriptive name
    2. *Prep Time*: How long to prepare
    3. *Cook Time*: How long to cook
    4. *Total Time*: Prep + Cook
    5. *Difficulty*: Easy / Medium / Hard
    6. *Servings*: How many people it serves
    7. *Ingredients*: Complete list with exact quantities (cups, tbsp, oz, etc.)
    8. *Instructions*: Numbered step-by-step cooking instructions (at least 5 steps)
    9. *Tips*: 1-2 helpful cooking tips

    Be specific and detailed. Each recipe should be complete enough to cook from.
    """,
    expected_output="""10 complete recipes, each with:
    - Name, times, difficulty, servings
    - Full ingredient list with quantities
    - Detailed step-by-step instructions
    - Cooking tips""",
    agent=researcher
)

#second agent - preference filter
filter = Agent(
    name="Filter",
    role="Preference Filter",
    goal="Filter recipes based on user's preferences and dislikes",
    backstory="Nutricionist who understands dietary needs. You check recipes against user's preferences and dislikes, and remove anything that does not match",
    tools=[SerperDevTool()],
    verbose=True
)

#second agent task
filter_task = Task(
    description=f"""Filter the recipes from the researcher based on user restrictions.
    ## REMOVE recipes containing:
    - Dislikes: {dislikes}
    - Allergies: {allergies}

    ## KEEP recipes that match:
    - Preferences: {preferences}

    ## RULES:
    - If a disliked ingredient can be easily removed/substituted, KEEP the recipe and note it
    - Allergies are strict - remove immediately if found
    - PRESERVE full recipe details (ingredients + instructions) for kept recipes

    ## OUTPUT FORMAT:

    Kept Recipes:
    For each kept recipe, include:
    - Recipe name
    - Why it passed the filter
    - Any substitution notes
    - FULL Context: (Prep Time, Cook Time, Total Time, Difficulty Servings, Tips)
    - FULL ingredients list
    - FULL instructions
    """,

    expected_output="List of filtered recipes with complete details preserved",
    agent=filter,
    context=[research_task]
)

#third agent - ranker
ranker = Agent(name="Ranker",
    role="Recipe Ranker",
    goal="Rank recipes by relevance, ease, and user match",
    backstory="Food critic with excellent taste and ranking abilities",
    tools=[SerperDevTool()],
    verbose=True
)
#third agent task
ranker_task = Task(
    description=f"""Rank the recipes from the filter based on user preferences, dislikes, and allergies.
    ## RANK recipes based on:
    - Dislikes: {dislikes}
    - Allergies: {allergies}
    - Preferences: {preferences}
    - Prep Time
    - Cook Time
    - Total Time
    - Difficulty
    - Servings

    ## SCORE:
    - Assign to every recipe a score from 0.0 to 10.0 being the best

    ## SELECT:
    - Top 3 recipes based on the score

    ## OUTPUT FORMAT:

    Selected Recipes:
    For each selected recipe, include:
    - Recipe name
    - Why the score was given
    - Any substitution notes
    - FULL Context: (Prep Time, Cook Time, Total Time, Difficulty Servings, Tips)
    - FULL ingredients list
    - FULL instructions
    - Score
    """,

    expected_output="List of selected recipes with complete details preserved",
    agent=ranker,
    context=[filter_task]
)


#fourth agent - skilled writer
writer = Agent(name="Writerr",
    role="Cookbook author",
    goal="Write recipes beginner-friendly",
    backstory="Cookbook author that writes recipes that are easy to read and understand no matter your cooking level",
    tools=[SerperDevTool()],
    verbose=True
)

#fourth agent task
writer_task = Task(
    description=f"""Write the recipes with clear headings, well-structured and bullets.""",
    expected_output="List of recipes, well-structured, aesthetic, and easy to follow using markdown and bullets format",
    agent=writer,
    context=[ranker_task]
)
# Assemble the crew and run tasks
crew = Crew(
    agents=[researcher, filter, ranker, writer],
    tasks=[research_task, filter_task, ranker_task, writer_task],
    process=Process.sequential,
)

result = crew.kickoff()

# Display the final report
display(Markdown(result.raw))